# DS 4320 Project 1 — Solution Pipeline
**New User Recommendation Model for Netflix**

**Author:** Rebecca Vanni

**NetID:** ecn2wh


### Setup and Packages

In [ ]:
# Install required packages if not already present
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import logging
import os
from sklearn.metrics.pairwise import cosine_similarity

# Configure logging to track pipeline execution
logging.basicConfig(
    filename='pipeline.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logging.info('Pipeline notebook started')
print('Setup complete.')

### Load Raw Data into DuckDB

Creating the four tables:

- `movies`: cleaned movie metadata with year extracted
- `ratings_agg`: per-movie average rating and count (min 50 ratings threshold)
- `genome_tags`: tag label lookup table
- `genome_scores`: tag relevance scores per movie (filtered to valid movies only)

In [ ]:
# Paths — raw CSVs are expected in /content (Colab default upload location)
RAW_DIR = '/content'
DB_PATH = '/content/movielens.db'
# Setting the threshold: ignore movies with fewer than 50 ratings (too sparse to trust)
MIN_RATINGS = 50

# Connect to DuckDB
try:
    con = duckdb.connect(DB_PATH)
    logging.info('Connected to DuckDB at %s', DB_PATH)

    # Drop tables if re-running so we start clean
    for t in ['genome_scores', 'genome_tags', 'ratings_agg', 'movies']:
        con.execute(f'DROP TABLE IF EXISTS {t}')

    # --- Table 1: movies ---
    # Extract release year from title using regex; drop rows with no genre
    con.execute(f"""
        CREATE TABLE movies AS
        SELECT
            CAST(movieId AS INTEGER) AS movieId,
            title,
            genres,
            TRY_CAST(regexp_extract(title, '\\((\\d{{4}})\\)$', 1) AS INTEGER) AS year
        FROM read_csv_auto('{RAW_DIR}/movies.csv')
        WHERE genres != '(no genres listed)'
    """)
    logging.info('movies created')

    # --- Table 2: ratings_agg ---
    # Aggregate 25M ratings into per-movie avg + count; apply MIN_RATINGS filter
    # ignore_errors=true skips the handful of malformed rows in the CSV
    con.execute(f"""
        CREATE TABLE ratings_agg AS
        SELECT
            CAST(movieId AS INTEGER) AS movieId,
            ROUND(AVG(rating), 4)    AS avg_rating,
            COUNT(*)                 AS rating_count
        FROM read_csv_auto('{RAW_DIR}/ratings.csv', ignore_errors=true)
        GROUP BY movieId
        HAVING COUNT(*) >= {MIN_RATINGS}
    """)
    logging.info('ratings_agg created')

    # Keep only movies that have enough ratings
    con.execute('DELETE FROM movies WHERE movieId NOT IN (SELECT movieId FROM ratings_agg)')

    # --- Table 3: genome_tags ---
    con.execute(f"""
        CREATE TABLE genome_tags AS
        SELECT
            CAST(tagId AS INTEGER) AS tagId,
            tag
        FROM read_csv_auto('{RAW_DIR}/genome-tags.csv')
    """)
    logging.info('genome_tags created')

    # --- Table 4: genome_scores ---
    # Filter to only movies that survived the earlier cleaning steps
    con.execute(f"""
        CREATE TABLE genome_scores AS
        SELECT
            CAST(gs.movieId   AS INTEGER) AS movieId,
            CAST(gs.tagId     AS INTEGER) AS tagId,
            CAST(gs.relevance AS DOUBLE)  AS relevance
        FROM read_csv_auto('{RAW_DIR}/genome-scores.csv') gs
        WHERE gs.movieId IN (SELECT movieId FROM movies)
    """)
    logging.info('genome_scores created')

except Exception as e:
    logging.error('Data load failed: %s', str(e))
    raise

### Query

The three queries used to extract data for the recommender:

- **Genre mapping**: explodes the pipe-delimited genre string into one row per genre per movie so we can match a user's survey responses.
- **Tag-genome pivot**: pulls the full relevance matrix for candidate movies (movies × tags). This is the content feature space for cosine similarity.
- **Popularity scores**: computes a normalized popularity score combining avg_rating and log(rating_count) to break ties between similarly relevant movies.

In [ ]:
# Genre mapping — explodes pipe-delimited genres into one row per movie-genre pair
# This allows filtering candidate movies by the user's stated genre preferences
try:
    genre_df = con.execute("""
        SELECT
            m.movieId,
            m.title,
            m.year,
            TRIM(genre_item) AS genre
        FROM movies m,
             UNNEST(string_split(m.genres, '|')) AS t(genre_item)
        WHERE TRIM(genre_item) != 'IMAX'
        ORDER BY m.movieId
    """).df()

    logging.info('genre_df: %d rows, %d unique movies', len(genre_df), genre_df['movieId'].nunique())
    print(f'genre_df: {len(genre_df):,} rows | {genre_df["movieId"].nunique():,} unique movies')
    print(f'Genres available: {sorted(genre_df["genre"].unique())}')
    genre_df.head()

except Exception as e:
    logging.error('Genre query failed: %s', str(e))
    raise

In [ ]:
# Pull genome scores and join with tag names
# This produces the movie x tag matrix needed for cosine similarity
try:
    genome_df = con.execute("""
        SELECT
            gs.movieId,
            gt.tag,
            gs.relevance
        FROM genome_scores gs
        JOIN genome_tags gt ON gs.tagId = gt.tagId
        ORDER BY gs.movieId, gt.tag
    """).df()

    logging.info('genome_df: %d rows', len(genome_df))
    print(f'genome_df: {len(genome_df):,} rows')
    genome_df.head()

except Exception as e:
    logging.error('Genome query failed: %s', str(e))
    raise

In [ ]:
# Compute popularity score
# 60% normalized avg_rating + 40% log-scaled rating_count
# Log-scaling prevents blockbusters from dominating while still rewarding popular titles
try:
    popularity_df = con.execute("""
        WITH scored AS (
            SELECT
                r.movieId,
                m.title,
                r.avg_rating,
                r.rating_count,
                (0.6 * (r.avg_rating / 5.0)) +
                (0.4 * (LN(r.rating_count) / LN((SELECT MAX(rating_count) FROM ratings_agg))))
                    AS popularity_score
            FROM ratings_agg r
            JOIN movies m ON r.movieId = m.movieId
        )
        SELECT
            movieId,
            title,
            avg_rating,
            rating_count,
            ROUND(popularity_score, 4) AS popularity_score
        FROM scored
        ORDER BY popularity_score DESC
    """).df()

    logging.info('popularity_df: %d rows', len(popularity_df))
    print(f'popularity_df: {len(popularity_df):,} movies')
    popularity_df.head(10)

except Exception as e:
    logging.error('Popularity query failed: %s', str(e))
    raise

### Model — Cold-Start Recommender

**Model Choice: Content-Based Filtering via Cosine Similarity on the Tag Genome**

Steps:
1. The user answers a one-question genre survey (selects 1–3 genres)
2. Candidate movies are identified by genre match
3. A **user profile vector** is built as the mean genome vector of the top-50 popular genre-matched movies
4. **Cosine similarity** is computed between the user profile and every candidate movie's genome vector
5. Final score = 70% cosine similarity + 30% popularity score
6. Top 5 results are returned as the recommendation list


In [ ]:
# Pivot the genome into a movie x tag matrix
# Rows = movies, Columns = tags, Values = relevance scores
# This is the core feature matrix for cosine similarity
try:
    genome_matrix = genome_df.pivot(index='movieId', columns='tag', values='relevance').fillna(0)
    logging.info('genome_matrix shape: %s', genome_matrix.shape)
    print(f'Genome matrix shape: {genome_matrix.shape}  (movies x tags)')
    genome_matrix.iloc[:3, :5]

except Exception as e:
    logging.error('Pivot failed: %s', str(e))
    raise

In [ ]:
def recommend_movies(user_genres, genre_df, genome_matrix, popularity_df,
                     top_n=5, sim_weight=0.7, pop_weight=0.3):
    """
    Cold-start recommender using cosine similarity on the tag genome.

    Parameters
    ----------
    user_genres   : list of str  — genres from onboarding survey
    genre_df      : DataFrame    — exploded movie-genre mapping
    genome_matrix : DataFrame    — movie x tag relevance matrix
    popularity_df : DataFrame    — pre-computed popularity scores
    top_n         : int          — number of recommendations (default 5)
    sim_weight    : float        — weight for cosine similarity in final score
    pop_weight    : float        — weight for popularity in final score

    Returns
    -------
    DataFrame with top_n recommended movies and their scores
    """
    try:
        # Step 1: Find candidate movies matching at least one user genre
        candidates = genre_df[genre_df['genre'].isin(user_genres)]['movieId'].unique()
        logging.info('Candidates for genres %s: %d movies', user_genres, len(candidates))

        if len(candidates) == 0:
            raise ValueError(f'No movies found for genres: {user_genres}')

        # Step 2: Build user profile from top-50 popular genre-matched movies
        # Using popular movies as the profile seed ensures we build a vector
        # representative of what most viewers in those genres enjoy, not outliers
        pop_candidates = (
            popularity_df[popularity_df['movieId'].isin(candidates)]
            .nlargest(50, 'popularity_score')['movieId']
            .values
        )

        # Filter to only those that have genome data
        seed_ids = [mid for mid in pop_candidates if mid in genome_matrix.index]
        if len(seed_ids) == 0:
            raise ValueError('No seed movies with genome data found.')

        user_profile = genome_matrix.loc[seed_ids].mean(axis=0).values.reshape(1, -1)
        logging.info('User profile built from %d seed movies', len(seed_ids))

        # Step 3: Compute cosine similarity between user profile and all candidates
        candidate_ids = [mid for mid in candidates if mid in genome_matrix.index]
        candidate_matrix = genome_matrix.loc[candidate_ids]
        sim_scores = cosine_similarity(user_profile, candidate_matrix.values)[0]
        sim_series = pd.Series(sim_scores, index=candidate_ids, name='cosine_sim')

        # Step 4: Normalize similarity scores to 0-1
        sim_min, sim_max = sim_series.min(), sim_series.max()
        sim_norm = (sim_series - sim_min) / (sim_max - sim_min + 1e-9)

        # Step 5: Normalize popularity and combine with similarity
        pop_lookup = popularity_df.set_index('movieId')['popularity_score']
        pop_norm = pop_lookup.reindex(candidate_ids).fillna(0)
        pop_norm = (pop_norm - pop_norm.min()) / (pop_norm.max() - pop_norm.min() + 1e-9)

        # Step 6: Final weighted score
        final_score = (sim_weight * sim_norm) + (pop_weight * pop_norm)

        # Step 7: Assemble results DataFrame
        results = pd.DataFrame({
            'movieId'    : candidate_ids,
            'cosine_sim' : sim_series.values,
            'pop_score'  : pop_lookup.reindex(candidate_ids).values,
            'final_score': final_score.values
        }).set_index('movieId')

        # Merge in title and rating info
        results = results.join(popularity_df.set_index('movieId')[['title', 'avg_rating', 'rating_count']])
        results = results.sort_values('final_score', ascending=False)

        # Remove seed movies from recommendations (don't recommend profile-builders)
        results = results[~results.index.isin(seed_ids)]

        logging.info('Recommendation complete. Returning top %d.', top_n)
        return results.head(top_n).reset_index()

    except Exception as e:
        logging.error('Recommendation failed: %s', str(e))
        raise

### Top-5 Recommendations

Simulating a new Netflix user completing the onboarding genre survey. The user selects their preferred genres and the model returns a ranked list of 5 movies.

In [ ]:
# Simulate the new user's onboarding genre survey
# In production this would come from the platform UI at signup
USER_GENRES = ['Action', 'Sci-Fi']   # <-- change these to try different users

recommendations = recommend_movies(
    user_genres   = USER_GENRES,
    genre_df      = genre_df,
    genome_matrix = genome_matrix,
    popularity_df = popularity_df,
    top_n         = 5,
    sim_weight    = 0.7,
    pop_weight    = 0.3
)

# Display clean results table
display_cols = ['title', 'avg_rating', 'rating_count', 'cosine_sim', 'final_score']
print('=== Top-5 Recommended Movies ===')
recommendations[display_cols].style.format({
    'avg_rating'   : '{:.2f}',
    'rating_count' : '{:,.0f}',
    'cosine_sim'   : '{:.4f}',
    'final_score'  : '{:.4f}'
})

### Visualization

**Score Breakdown Bar Chart:** A horizontal stacked bar chart was chosen to make the model's reasoning transparent. This shows exactly how much each recommendation's final score comes from content similarity versus popularity so the reader can see the model is not simply recommending the most popular movies. Netflix brand colors were used to ground the visualization in the platform context and distinguish the two score components clearly.

**Tag Genome Heatmap:** A heatmap was chosen to display the 20 most variable tags across the top-5 recommended movies because it simultaneously reveals both which content features drove each recommendation and how the five movies differ from one another along those features. Variance-based tag selection ensures the heatmap shows the most discriminating features rather than tags that are uniformly high or low across all recommendations, making the content-based reasoning interpretable at a glance.

In [ ]:
# Score breakdown bar chart
# Shows how much each component (content similarity vs popularity) contributes
sns.set_theme(style='whitegrid', font_scale=1.1)
BRAND_RED  = '#E50914'   # Netflix red
BRAND_DARK = '#221F1F'   # Netflix dark
BRAND_GRAY = '#B3B3B3'

fig, ax = plt.subplots(figsize=(10, 5))
titles   = recommendations['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True)
sim_vals = recommendations['cosine_sim']
pop_vals = recommendations['pop_score']
y_pos    = range(len(titles))

# Stacked horizontal bars — similarity component + popularity component
bars1 = ax.barh(y_pos, sim_vals * 0.7, color=BRAND_RED,  label='Content Similarity (70%)', height=0.55)
bars2 = ax.barh(y_pos, pop_vals * 0.3, left=sim_vals * 0.7,
                color=BRAND_GRAY, label='Popularity Score (30%)', height=0.55)

# Rank labels on the left
ax.set_yticks(list(y_pos))
ax.set_yticklabels([f'#{i+1}  {t}' for i, t in enumerate(titles)], fontsize=11)
ax.invert_yaxis()
ax.set_xlabel('Weighted Score Contribution', fontsize=11)
ax.set_title(
    f'Top-5 Netflix Recommendations\nNew User Genres: {", ".join(USER_GENRES)}',
    fontsize=13, fontweight='bold', color=BRAND_DARK, pad=12
)
ax.legend(loc='lower right', framealpha=0.9)
ax.set_facecolor('#F9F9F9')
fig.patch.set_facecolor('white')

# Add avg rating annotation
for i, row in recommendations.iterrows():
    ax.text(
        row['final_score'] + 0.002, i,
        f'★ {row["avg_rating"]:.2f}  ({row["rating_count"]/1000:.0f}K ratings)',
        va='center', fontsize=9, color=BRAND_DARK
    )

plt.tight_layout()
plt.savefig('recommendation_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to recommendation_scores.png')

In [ ]:
# Tag Relevance Heatmap for Top-5 Movies
# Shows which content features drove each recommendation
# Variance-based tag selection highlights the most discriminating features

top_movie_ids = recommendations['movieId'].tolist()
top_titles = dict(zip(recommendations['movieId'],
                      recommendations['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True)))

# Get genome vectors for just our top-5
heat_data = genome_matrix.loc[[mid for mid in top_movie_ids if mid in genome_matrix.index]]
heat_data.index = [top_titles[mid] for mid in heat_data.index]

# Pick the top 20 tags by variance across these 5 movies (most informative)
top_tags  = heat_data.var(axis=0).nlargest(20).index
heat_data = heat_data[top_tags]

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    heat_data,
    ax=ax,
    cmap='Reds',
    linewidths=0.4,
    linecolor='#EEEEEE',
    cbar_kws={'label': 'Tag Relevance Score', 'shrink': 0.8},
    annot=True,
    fmt='.2f',
    annot_kws={'size': 8}
)

ax.set_title(
    'Tag Genome Relevance — Top-5 Recommended Movies\n(Top 20 most variable tags shown)',
    fontsize=12, fontweight='bold', pad=12
)
ax.set_xlabel('Tag', fontsize=10)
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0,  labelsize=9)

plt.tight_layout()
plt.savefig('tag_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved to tag_heatmap.png')

### Analysis Rationale

The cold-start recommendation model uses content-based filtering via cosine similarity on the MovieLens tag genome, rather than collaborative filtering, because new users have no watch history to compute user-user or item-item similarity against. Final scores blend 70% content similarity with 30% log-normalized popularity to ensure recommendations are both personally relevant to the user's stated genre preferences and well-regarded by the broader platform community.

| Decision | Choice | Rationale |
|---|---|---|
| Filtering method | Content-based cosine similarity | Cold-start: no user history for collaborative filtering |
| Feature space | Tag genome (1,128 tags) | Precomputed, dense, semantically rich |
| User profile | Mean vector of top-50 popular genre-matched movies | Stable representative starting point |
| Score weighting | 70% similarity + 30% popularity | Prioritizes content match while ensuring well-regarded titles |
| Popularity normalization | Log-scaled rating count | Prevents blockbusters from dominating; rewards quality over volume |
| Rating threshold | Min 50 ratings | Removes movies too sparse to trust |

### Limitations
- The tag genome covers ~13,000 of 62,000 movies — movies without genome data are excluded from content ranking.
- The user profile is bootstrapped from popularity, not true taste — appropriate for cold-start but should update as watch history grows.
- Genre labels are coarse; the tag genome partially compensates for sub-genre variation.


In [ ]:
# Close database connection cleanly
con.close()
logging.info('Pipeline notebook complete. DB connection closed.')
print('Pipeline complete. Database connection closed.')